In [1]:
!pip install langchain_openai langchain_chroma langchain_community load_dotenv faiss-cpu sentence-transformers pymupdf GitPython

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.4/476.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.4 MB/s eta 0:00:00


In [2]:
from dotenv import load_dotenv
load_dotenv('./drive/MyDrive/data/apikeyfile')

True

In [3]:
from langchain_community.tools.tavily_search import TavilySearchResults

tools = [TavilySearchResults(max_results=3)]

/tmp/ipython-input-2518516040.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tools = [TavilySearchResults(max_results=3)]


In [4]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(model='gpt-4o')
agent_executor = create_agent(llm, tools)

In [5]:
resp = agent_executor.invoke({'messages':[('user', 'LangChain의 개요를 알려줘')]})

In [9]:
resp['messages'][-1].content

'LangChain은 자연어 처리(NLP)와 인공지능(AI) 응용 프로그램을 개발하기 위한 프레임워크입니다. 주로 언어 모델(Language Model)을 활용하여 다양한 AI 시스템을 구축하는 데 중점을 둡니다. LangChain은 언어 모델을 기반으로 한 체인을 만들어, 각각의 구성 요소들이 상호작용하여 복잡한 작업을 수행할 수 있도록 구조화합니다.\n\n### 주요 특징:\n1. **모듈성**: LangChain은 다양한 모듈을 제공하여, 텍스트 생성, 요약, 질의응답 등 여러 NLP 기능을 손쉽게 구현할 수 있습니다.\n2. **확장성**: 사용자가 자신만의 모듈이나 기능을 추가하고 확장할 수 있는 유연성을 제공합니다.\n3. **통합**: 다른 AI 서비스 및 데이터 소스와의 통합을 지원하여, 보다 풍부한 기능과 데이터를 활용할 수 있습니다.\n4. **체인**: 복잡한 작업을 여러 단계로 나누어 처리할 수 있는 체인 구조를 지원합니다. 이는 작업의 추상화 수준을 높이고 재사용성을 향상시킵니다.\n\nLangChain은 이러한 특징을 통해 사용자가 효율적으로 AI 모델을 활용하고, 복잡한 언어 기반 응용 프로그램을 보다 쉽게 개발할 수 있도록 지원합니다.'

In [10]:
from langchain_community.document_loaders import GitLoader


def file_filter(file_path: str) -> bool:
    return file_path.endswith(".md")


loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter,
)

documents = loader.load()
print(len(documents))

36


In [11]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

documents = text_splitter.split_documents(documents)
print(len(documents))


96


In [12]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_community.embeddings import HuggingFaceEmbeddings

In [13]:
embeddings_model = HuggingFaceEmbeddings(
    model_name='jhgan/ko-sbert-nli',
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings':True},
)


vectorstore = FAISS.from_documents(documents,
                                   embedding = embeddings_model,
                                   distance_strategy = DistanceStrategy.COSINE
                                  )

/tmp/ipython-input-707567803.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
query = 'LangChain의 개요를 알려줘'

In [15]:
retriever = vectorstore.as_retriever(search_kwargs={'k':1})
docs = retriever.invoke(query)
print(len(docs))
print(docs[0])

1
page_content='LangChain is the easiest way to start building agents and applications powered by LLMs. With under 10 lines of code, you can connect to OpenAI, Anthropic, Google, and [more](https://docs.langchain.com/oss/python/integrations/providers/overview). LangChain provides a pre-built agent architecture and model integrations to help you get started quickly and seamlessly incorporate LLMs into your agents and applications.

We recommend you use LangChain if you want to quickly build agents and autonomous applications. Use [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview), our low-level agent orchestration framework and runtime, when you have more advanced needs that require a combination of deterministic and agentic workflows, heavy customization, and carefully controlled latency.' metadata={'source': 'libs/langchain_v1/README.md', 'file_path': 'libs/langchain_v1/README.md', 'file_name': 'README.md', 'file_type': '.md'}


In [16]:
retriever = vectorstore.as_retriever(search_type='mmr', search_kwargs={'k':5, 'fetch_k':50})
docs = retriever.invoke(query)
print(len(docs))
print(docs[0])

5
page_content='LangChain is the easiest way to start building agents and applications powered by LLMs. With under 10 lines of code, you can connect to OpenAI, Anthropic, Google, and [more](https://docs.langchain.com/oss/python/integrations/providers/overview). LangChain provides a pre-built agent architecture and model integrations to help you get started quickly and seamlessly incorporate LLMs into your agents and applications.

We recommend you use LangChain if you want to quickly build agents and autonomous applications. Use [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview), our low-level agent orchestration framework and runtime, when you have more advanced needs that require a combination of deterministic and agentic workflows, heavy customization, and carefully controlled latency.' metadata={'source': 'libs/langchain_v1/README.md', 'file_path': 'libs/langchain_v1/README.md', 'file_name': 'README.md', 'file_type': '.md'}


lambda 가 작을 수록 중복을 줄이고 다양하게 추출

In [17]:
retriever = vectorstore.as_retriever(search_type='mmr', search_kwargs={'k':5, 'lambda_mult':0.15})
docs = retriever.invoke(query)
print(len(docs))
print(docs[-1])

5
page_content='- **Core layer** (`langchain-core`): Base abstractions, interfaces, and protocols. Users should not need to know about this layer directly.
- **Implementation layer** (`langchain`): Concrete implementations and high-level public utilities
- **Integration layer** (`partners/`): Third-party service integrations. Note that this monorepo is not exhaustive of all LangChain integrations; some are maintained in separate repos, such as `langchain-ai/langchain-google` and `langchain-ai/langchain-aws`. Usually these repos are cloned at the same level as this monorepo, so if needed, you can refer to their code directly by navigating to `../langchain-google/` from this monorepo.
- **Testing layer** (`standard-tests/`): Standardized integration tests for partner integrations

### Development tools & commands**' metadata={'source': 'AGENTS.md', 'file_path': 'AGENTS.md', 'file_name': 'AGENTS.md', 'file_type': '.md'}


In [18]:
retriever = vectorstore.as_retriever(search_type='similarity_score_threshold', search_kwargs={'score_threshold':0.1})
docs = retriever.invoke(query)
print(len(docs))

4


In [19]:
result = vectorstore.similarity_search_with_score(query=query, k=4)

In [22]:
for doc, score in result:
    print(score)

0.73375213
0.7413166
0.7413166
0.7691163


In [23]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Retrieval
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5, 'lambda_mult': 0.15}
)

docs = retriever.invoke(query)

# Prompt
template = '''Answer the question based only on the following context:
{context}

Question: {question}
'''

prompt = ChatPromptTemplate.from_template(template)

# Model
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0,
    max_tokens=500,
)


def format_docs(docs):
    return '\n\n'.join([d.page_content for d in docs])

# Chain
chain = prompt | llm | StrOutputParser()

# Run
response = chain.invoke({'context': (format_docs(docs)), 'question':query})
response

'LangChain은 LLM(대형 언어 모델)으로 구동되는 에이전트 및 애플리케이션을 쉽게 구축할 수 있는 방법입니다. 10줄도 안 되는 코드로 OpenAI, Anthropic, Google 등과 연결할 수 있으며, 사전 구축된 에이전트 아키텍처와 모델 통합을 제공하여 빠르게 시작하고 LLM을 애플리케이션에 원활하게 통합할 수 있도록 돕습니다. LangChain을 사용하면 에이전트와 자율 애플리케이션을 신속하게 구축할 수 있으며, 더 복잡한 요구 사항이 있는 경우에는 LangGraph라는 저수준 에이전트 오케스트레이션 프레임워크를 사용할 수 있습니다.'

In [24]:
from langchain_community.document_loaders import GitLoader


def file_filter(file_path: str) -> bool:
    return file_path.endswith(".md")


loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter,
)

documents = loader.load()
print(len(documents))

36


In [25]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

docs = text_splitter.split_documents(documents)

In [26]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [27]:
from langchain_chroma import Chroma

db = Chroma.from_documents(docs, embeddings)

In [28]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [29]:
retriever = db.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5, 'lambda_mult': 0.15}
)

docs = retriever.invoke(query)

In [30]:
# Prompt
template = '''Answer the question based only on the following context:
{context}

Question: {question}
'''

prompt = ChatPromptTemplate.from_template(template)

# Model
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0,
    max_tokens=500,
)

In [31]:
query = 'LangChain의 개요를 알려줘'

In [32]:
def format_docs(docs):
    return '\n\n'.join([d.page_content for d in docs])

# Chain
chain = prompt | llm | StrOutputParser()

# Run
response = chain.invoke({'context': (format_docs(docs)), 'question':query})
response

'LangChain은 에이전트 및 LLM 기반 애플리케이션을 구축하기 위한 프레임워크입니다. 이 프레임워크는 상호 운용 가능한 구성 요소와 제3자 통합을 연결하여 AI 애플리케이션 개발을 간소화하며, 기술이 발전함에 따라 미래에 대비할 수 있는 결정을 내릴 수 있도록 돕습니다. LangChain은 모듈화, 안정성, 그리고 검증된 성능을 제공하는 `langchain-core`를 기반으로 하여, 다양한 모델 제공자가 요구하는 인터페이스를 구현할 수 있도록 설계되었습니다. 이를 통해 LangChain 생태계의 나머지 부분에서 쉽게 사용할 수 있습니다.'

In [33]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

In [36]:
openai_model: str = "gpt-4o"
temperature: float = 0.0

In [34]:
class Goal(BaseModel):
    description: str = Field(..., description="목표 설명")

    @property
    def text(self) -> str:
        return f"{self.description}"

In [38]:
class PassiveGoalCreator:
    def __init__(self,llm: ChatOpenAI,):
        self.llm = llm

    def run(self, query:str) -> Goal:
        prompt = ChatPromptTemplate.from_template(
            "사용자 입력을 분석하여 명확하고 실행 가능한 목표를 생성해 주세요.\n"
            "요건:\n"
            "1. 목표는 구체적이고 명확해야 하며, 실행 가능한 수준으로 상세화되어야 합니다.\n"
            "2. 당신이 실행할 수 있는 행동은 다음과 같은 행동뿐입니다.\n"
            "   - 인터넷을 이용하여 목표 달성을 위한 조사를 수행합니다.\n"
            "   - 사용자를 위한 보고서를 생성합니다.\n"
            "3. 절대 2.에 명시된 행동 외의 다른 행동을 취해서는 안 됩니다.\n"
            "사용자 입력: {query}"
        )
        chain = prompt | self.llm.with_structured_output(Goal)
        return chain.invoke({'query':query})

In [35]:
task = '삼성전자 주식에 대해 평가해줘'

In [39]:
llm = ChatOpenAI(model= openai_model, temperature=temperature)
goal_creator = PassiveGoalCreator(llm=llm)
result: Goal = goal_creator.run(query=task)
print(f"{result.text}")

### 목표: 삼성전자 주식 평가 보고서 작성

#### 1. 목표 설명
삼성전자 주식에 대한 포괄적이고 명확한 평가 보고서를 작성합니다. 이 보고서는 삼성전자의 현재 주식 상태, 시장 동향, 경쟁사 비교, 그리고 미래 전망을 포함해야 합니다.

#### 2. 목표 달성을 위한 세부 단계

1. **삼성전자 주식의 현재 상태 조사**
   - 삼성전자의 최근 주가 변동 및 거래량 분석
   - 최근 분기 실적 및 재무제표 검토
   - 주요 뉴스 및 발표 사항 확인

2. **시장 동향 및 경쟁사 분석**
   - 반도체 및 전자 산업의 최근 트렌드 조사
   - 주요 경쟁사(예: 인텔, TSMC)와의 비교 분석
   - 글로벌 경제 상황이 삼성전자에 미치는 영향 평가

3. **미래 전망 및 전문가 의견 수집**
   - 주식 전문가 및 애널리스트의 삼성전자 주식에 대한 의견 수집
   - 삼성전자의 미래 계획 및 전략적 방향성 조사
   - 기술 혁신 및 신제품 출시 계획 확인

4. **보고서 작성 및 사용자 제공**
   - 수집한 정보를 바탕으로 명확하고 체계적인 보고서 작성
   - 보고서에 삼성전자 주식의 강점, 약점, 기회, 위협(SWOT 분석) 포함
   - 사용자에게 보고서 제공 및 추가 질문에 대한 답변 준비

#### 3. 목표 달성의 기대 결과
- 삼성전자 주식에 대한 명확하고 실행 가능한 평가 보고서가 완성됩니다.
- 사용자는 삼성전자 주식에 대한 깊이 있는 이해를 바탕으로 투자 결정을 내릴 수 있습니다.


In [40]:
class OptimizedGoal(BaseModel):
    description: str = Field(..., description="목표 설명")
    metrics: str = Field(..., description="목표의 달성도를 측정하는 방법")

    @property
    def text(self) -> str:
        return f"{self.description}(측정 기준:{self.metrics})"

In [41]:
class PromptOptimizer:
    def __init__(self,llm: ChatOpenAI,):
        self.llm = llm

    def run(self, query:str) -> OptimizedGoal:
        prompt = ChatPromptTemplate.from_template(
            "당신은 목표 설정 전문가입니다. 아래의 목표를 SMART 원칙(Specific: 구체적, Measurable: 측정 가능, Achievable: 달성 가능, Relevant: 관련성이 높은, Time-bound: 기한이 있는)에 기반하여 최적화해 주세요.\n\n"
            "원래 목표:\n"
            "{query}\n\n"
            "지시 사항:\n"
            "1. 원래 목표를 분석하고, 부족한 요소나 개선점을 파악해 주세요.\n"
            "2. 당신이 실행할 수 있는 행동은 다음과 같습니다.\n"
            "   - 인터넷을 이용하여 목표 달성을 위한 조사를 수행한다.\n"
            "   - 사용자를 위한 보고서를 생성한다.\n"
            "3. SMART 원칙의 각 요소를 고려하면서 목표를 구체적이고 상세하게 기술해 주세요.\n"
            "   - 절대 추상적인 표현을 포함해서는 안 됩니다.\n"
            "   - 반드시 모든 단어가 실행 가능하고 구체적인지 확인해 주세요.\n"
            "4. 목표의 달성도를 측정하는 방법을 구체적이고 상세하게 기술해 주세요.\n"
            "5. 원래 목표에서 기한이 지정되지 않은 경우에는 기한을 고려할 필요가 없습니다.\n"
            "6. 주의: 절대로 2번 이외의 행동을 취해서는 안 됩니다.")
        chain = prompt | self.llm.with_structured_output(OptimizedGoal)
        return chain.invoke({'query':query})

In [42]:
llm = ChatOpenAI(model= openai_model, temperature=temperature)
passive_goal_creator = PassiveGoalCreator(llm=llm)
goal: Goal = passive_goal_creator.run(query=task)

prompt_optimizer = PromptOptimizer(llm=llm)
optimized_goal: OptimizedGoal = prompt_optimizer.run(query=goal.text)
print(f"{optimized_goal.text}")

### 최적화된 목표: 삼성전자 주식 평가 보고서 작성

#### 1. 목표 설명
삼성전자 주식에 대한 포괄적이고 명확한 평가 보고서를 작성하여, 투자자들이 삼성전자 주식에 대한 정보에 기반한 결정을 내릴 수 있도록 지원합니다. 이 보고서는 삼성전자의 현재 주식 상태, 시장 동향, 경쟁사 비교, 그리고 미래 전망을 포함합니다.

#### 2. 목표 달성을 위한 세부 단계

1. **삼성전자 주식의 현재 상태 조사**
   - 삼성전자의 최근 6개월간 주가 변동 및 거래량 분석
   - 최근 2분기 실적 및 재무제표 검토
   - 지난 3개월간 주요 뉴스 및 발표 사항 확인

2. **시장 동향 및 경쟁사 분석**
   - 지난 1년간 반도체 및 전자 산업의 주요 트렌드 조사
   - 주요 경쟁사(예: SK하이닉스, 인텔 등)와의 최근 1년간 비교 분석
   - 최근 6개월간 글로벌 경제 및 정치적 요인들이 삼성전자에 미치는 영향 평가

3. **미래 전망 및 전문가 의견 수집**
   - 삼성전자의 향후 1년간 기술 개발 및 사업 전략 분석
   - 주식 시장 전문가 및 애널리스트의 최근 3개월간 의견 및 예측 수집
   - 장기적인 투자 가치 평가

4. **보고서 작성 및 사용자 제공**
   - 수집한 정보를 바탕으로 20페이지 내외의 명확하고 체계적인 보고서 작성
   - 보고서에는 삼성전자의 강점, 약점, 기회, 위협(SWOT 분석) 포함
   - 사용자에게 보고서 제공 및 추가 질문에 대한 답변 준비

#### 3. 목표 달성의 기대 결과
- 삼성전자 주식에 대한 명확하고 실행 가능한 평가 보고서가 완성됩니다.
- 사용자는 삼성전자 주식에 대한 투자 결정을 내리는 데 필요한 정보를 얻게 됩니다.

#### 4. 목표 달성의 측정 방법
- 보고서의 완성도: 모든 세부 단계가 포함된 보고서가 작성되었는지 확인
- 사용자 피드백: 보고서를 받은 사용자의 피드백을 통해 정보의 유용성과 명확성을 평가
- 보고서 제공 후 1개월 내에 사용자로부터 추가 질문이 